В цьому домашньому завданні ми знову працюємо з даними з нашого змагання ["Bank Customer Churn Prediction (DLU Course)"](https://www.kaggle.com/t/7c080c5d8ec64364a93cf4e8f880b6a0).

Тут ми побудуємо рішення задачі класифікації з використанням алгоритмів бустингу: XGBoost та LightGBM, а також використаємо бібліотеку HyperOpt для оптимізації гіперпараметрів.

0. Зчитайте дані `train.csv` в змінну `raw_df` та скористайтесь наведеним кодом нижче аби розділити дані на трнувальні та валідаційні і розділити дані на ознаки з матириці Х та цільову змінну. Назви змінних `train_inputs, train_targets, train_inputs, train_targets` можна змінити на ті, які Вам зручно.

  Наведений скрипт - частина отриманого мною скрипта для обробки даних. Ми тут не викнуємо масштабування та обробку категоріальних змінних, бо хочемо це делегувати алгоритмам, які будемо використовувати. Якщо щось не розумієте в наведених скриптах, рекомендую розібратись: навичка читати код - важлива складова роботи в машинному навчанні.

In [1]:
from xgboost import XGBClassifier
from sklearn.metrics import classification_report
from sklearn.metrics import roc_auc_score

In [2]:
! pip freeze | grep xgboost

xgboost==3.3.0


In [3]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from typing import Tuple, Dict, Any


def split_train_val(df: pd.DataFrame, target_col: str, test_size: float = 0.2, random_state: int = 42) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Split the dataframe into training and validation sets.

    Args:
        df (pd.DataFrame): The raw dataframe.
        target_col (str): The target column for stratification.
        test_size (float): The proportion of the dataset to include in the validation split.
        random_state (int): Random state for reproducibility.

    Returns:
        Tuple[pd.DataFrame, pd.DataFrame]: Training and validation dataframes.
    """
    train_df, val_df = train_test_split(df, test_size=test_size, random_state=random_state, stratify=df[target_col])
    return train_df, val_df


def separate_inputs_targets(df: pd.DataFrame, input_cols: list, target_col: str) -> Tuple[pd.DataFrame, pd.Series]:
    """
    Separate inputs and targets from the dataframe.

    Args:
        df (pd.DataFrame): The dataframe.
        input_cols (list): List of input columns.
        target_col (str): Target column.

    Returns:
        Tuple[pd.DataFrame, pd.Series]: DataFrame of inputs and Series of targets.
    """
    inputs = df[input_cols].copy()
    targets = df[target_col].copy()
    return inputs, targets

In [4]:
raw_df = pd.read_csv("train.csv")

train_df, val_df = split_train_val(
    raw_df,
    target_col="Exited"
)

input_cols = [
    col for col in raw_df.columns
    if col not in ["Exited", "CustomerId", "Surname"]
]

train_inputs, train_targets = separate_inputs_targets(
    train_df,
    input_cols,
    "Exited"
)

val_inputs, val_targets = separate_inputs_targets(
    val_df,
    input_cols,
    "Exited"
)

print("Train inputs:", train_inputs.shape)
print("Train targets:", train_targets.shape)
print("Validation inputs:", val_inputs.shape)
print("Validation targets:", val_targets.shape)

Train inputs: (12000, 11)
Train targets: (12000,)
Validation inputs: (3000, 11)
Validation targets: (3000,)


1. В тренувальному та валідаційному наборі перетворіть категоріальні ознаки на тип `category`. Можна це зробити двома способами:
 1. `df[col_name].astype('category')`, як було продемонстровано в лекції
 2. використовуючи метод `pd.Categorical(df[col_name])`

In [5]:
categorical_cols = train_inputs.select_dtypes(include="object").columns.tolist()
print(categorical_cols)

['Geography', 'Gender']


In [6]:
for col in categorical_cols:
    train_inputs[col] = train_inputs[col].astype("category")
    val_inputs[col] = val_inputs[col].astype("category")

In [7]:
train_inputs.dtypes

,0
id,int64
CreditScore,float64
Geography,category
Gender,category
Age,float64
Tenure,float64
Balance,float64
NumOfProducts,float64
HasCrCard,float64
IsActiveMember,float64


2. Навчіть на отриманих даних модель `XGBoostClassifier`. Параметри алгоритму встановіть на свій розсуд, ми далі будемо їх тюнити. Рекомендую тренувати не дуже складну модель.

  Опис всіх конфігураційних параметрів XGBoostClassifier - тут https://xgboost.readthedocs.io/en/stable/parameter.html#global-config

  **Важливо:** зробіть такі налаштування `XGBoostClassifier` аби він самостійно обробляв незаповнені значення в даних і обробляв категоріальні колонки.

  Можна також, якщо працюєте в Google Colab, увімкнути можливість використання GPU (`Runtime -> Change runtime type -> T4 GPU`) і встановити параметр `device='cuda'` в `XGBoostClassifier` для пришвидшення тренування бустинг моделі.
  
  Після тренування моделі
  1. Виміряйте точність з допомогою AUROC на тренувальному та валідаційному наборах.
  2. Зробіть висновок про отриману модель: вона хороша/погана, чи є high bias/high variance?
  3. Порівняйте якість цієї моделі з тою, що ви отрмали з використанням DecisionTrees раніше. Чи вийшло покращити якість?

In [8]:
xgb_model = XGBClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    enable_categorical=True,
    tree_method="hist"
)

xgb_model.fit(
    train_inputs,
    train_targets
)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=True, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.05, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=4,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=200,
              n_jobs=None, num_parallel_tree=None, ...)

In [9]:
train_probs_xgb = xgb_model.predict_proba(train_inputs)[:, 1]
val_probs_xgb = xgb_model.predict_proba(val_inputs)[:, 1]

train_auc_xgb = roc_auc_score(train_targets, train_probs_xgb)
val_auc_xgb = roc_auc_score(val_targets, val_probs_xgb)

print(f"Train AUROC: {train_auc_xgb:.4f}")
print(f"Validation AUROC: {val_auc_xgb:.4f}")

Train AUROC: 0.9519
Validation AUROC: 0.9356


Отримана модель XGBoost має хороший результат: AUROC на тренувальній вибірці становить 0.9519, а на валідаційній — 0.9356. Різниця між показниками становить лише 0.0163, тому модель не має значних ознак перенавчання (high variance) та добре узагальнює дані.

У порівнянні з Decision Tree модель XGBoost показала кращий результат на валідаційній вибірці. Validation AUROC зріс з 0.9176 для найкращого Decision Tree з попередніх експериментів до 0.9356 для XGBoost. Отже, використання бустингу дозволило покращити якість класифікації.

3. Використовуючи бібліотеку `Hyperopt` і приклад пошуку гіперпараметрів для `XGBoostClassifier` з лекції знайдіть оптимальні значення гіперпараметрів `XGBoostClassifier` для нашої задачі. Задайте свою сітку гіперпараметрів виходячи з тих параметрів, які ви б хотіли перебрати. Поставте кількість раундів в підборі гіперпараметрів рівну **20**.

  **Увага!** Для того, аби скористатись hyperopt, нам треба задати функцію `objective`. В ній ми маємо задати loss - це може будь-яка метрика, але бажано використовувтаи ту, яка цільова в вашій задачі. Чим менший лосс - тим ліпша модель на думку hyperopt. Тож, тут нам треба задати loss - негативне значення AUROC. В лекції ми натомість використовували Accuracy.

  Після успішного завершення пошуку оптимальних гіперпараметрів
    - виведіть найкращі значення гіперпараметрів
    - створіть в окремій зміній `final_clf` модель `XGBoostClassifier` з найкращими гіперпараметрами
    - навчіть модель `final_clf`
    - оцініть якість моделі `final_clf` на тренувальній і валідаційній вибірках з допомогою AUROC.
    - зробіть висновок про якість моделі. Чи стала вона краще порівняно з попереднім пунктом (2) цього завдання?

In [10]:
!pip install hyperopt

In [11]:
from hyperopt import hp, fmin, tpe, Trials, STATUS_OK

In [12]:
space = {
    "n_estimators": hp.quniform("n_estimators", 100, 500, 50),
    "max_depth": hp.quniform("max_depth", 2, 8, 1),
    "learning_rate": hp.uniform("learning_rate", 0.01, 0.2),
    "subsample": hp.uniform("subsample", 0.6, 1.0),
    "colsample_bytree": hp.uniform("colsample_bytree", 0.6, 1.0),
    "min_child_weight": hp.quniform("min_child_weight", 1, 10, 1),
    "gamma": hp.uniform("gamma", 0, 2),
}

In [13]:
def objective(params):
    params["n_estimators"] = int(params["n_estimators"])
    params["max_depth"] = int(params["max_depth"])
    params["min_child_weight"] = int(params["min_child_weight"])

    model = XGBClassifier(
        **params,
        enable_categorical=True,
        tree_method="hist",
        random_state=42
    )

    model.fit(
        train_inputs,
        train_targets
    )

    val_probs = model.predict_proba(val_inputs)[:, 1]
    val_auc = roc_auc_score(val_targets, val_probs)

    return {
        "loss": -val_auc,
        "status": STATUS_OK
    }

In [14]:
trials = Trials()

best = fmin(
    fn=objective,
    space=space,
    algo=tpe.suggest,
    max_evals=20,
    trials=trials,
    rstate=np.random.default_rng(42)
)

100%|██████████| 20/20 [00:45<00:00,  2.28s/trial, best loss: -0.935488030729131]


In [15]:
print("Best hyperparameters:")
print(best)

Best hyperparameters:
{'colsample_bytree': np.float64(0.6823764671265532), 'gamma': np.float64(1.0015012197901458), 'learning_rate': np.float64(0.02404414361184864), 'max_depth': np.float64(6.0), 'min_child_weight': np.float64(6.0), 'n_estimators': np.float64(200.0), 'subsample': np.float64(0.7615110681047916)}


In [16]:
best["n_estimators"] = int(best["n_estimators"])
best["max_depth"] = int(best["max_depth"])
best["min_child_weight"] = int(best["min_child_weight"])

In [17]:
final_clf = XGBClassifier(
    **best,
    enable_categorical=True,
    tree_method="hist",
    random_state=42
)

In [18]:
final_clf.fit(
    train_inputs,
    train_targets
)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=np.float64(0.6823764671265532), device=None,
              early_stopping_rounds=None, enable_categorical=True,
              eval_metric=None, feature_types=None, feature_weights=None,
              gamma=np.float64(1.0015012197901458), grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=np.float64(0.02404414361184864), max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=6, max_leaves=None,
              min_child_weight=6, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=200, n_jobs=None,
              num_parallel_tree=None, ...)

In [19]:
train_probs_final = final_clf.predict_proba(train_inputs)[:, 1]
val_probs_final = final_clf.predict_proba(val_inputs)[:, 1]

train_auc_final = roc_auc_score(
    train_targets,
    train_probs_final
)

val_auc_final = roc_auc_score(
    val_targets,
    val_probs_final
)

print(f"Train AUROC: {train_auc_final:.4f}")
print(f"Validation AUROC: {val_auc_final:.4f}")

Train AUROC: 0.9562
Validation AUROC: 0.9355


модель після Hyperopt не стала кращою за попередню, якщо оцінювати за Validation AUROC

4. Навчіть на наших даних модель LightGBM. Параметри алгоритму встановіть на свій розсуд, ми далі будемо їх тюнити. Рекомендую тренувати не дуже складну модель.

  Опис всіх конфігураційних параметрів LightGBM - тут https://lightgbm.readthedocs.io/en/latest/Parameters.html

  **Важливо:** зробіть такі налаштування LightGBM аби він самостійно обробляв незаповнені значення в даних і обробляв категоріальні колонки.

  Аби передати категоріальні колонки в LightGBM - необхідно виявити їх індекси і передати в параметрі `cat_feature=cat_feature_indexes`

  Після тренування моделі
  1. Виміряйте точність з допомогою AUROC на тренувальному та валідаційному наборах.
  2. Зробіть висновок про отриману модель: вона хороша/погана, чи є high bias/high variance?
  3. Порівняйте якість цієї моделі з тою, що ви отрмали з використанням XGBoostClassifier раніше. Чи вийшло покращити якість?

In [20]:
from lightgbm import LGBMClassifier

In [21]:
cat_feature_indexes = [
    i for i, col in enumerate(train_inputs.columns)
    if train_inputs[col].dtype.name == "category"
]

print(cat_feature_indexes)

[2, 3]


In [22]:
lgbm_model = LGBMClassifier(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=4,
    num_leaves=15,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    verbosity=-1
)

In [23]:
lgbm_model.fit(
    train_inputs,
    train_targets,
    categorical_feature=cat_feature_indexes
)

LGBMClassifier(colsample_bytree=0.8, learning_rate=0.05, max_depth=4,
               n_estimators=200, num_leaves=15, random_state=42, subsample=0.8,
               verbosity=-1)

In [24]:
train_probs_lgbm = lgbm_model.predict_proba(train_inputs)[:, 1]
val_probs_lgbm = lgbm_model.predict_proba(val_inputs)[:, 1]

train_auc_lgbm = roc_auc_score(
    train_targets,
    train_probs_lgbm
)

val_auc_lgbm = roc_auc_score(
    val_targets,
    val_probs_lgbm
)

print(f"Train AUROC: {train_auc_lgbm:.4f}")
print(f"Validation AUROC: {val_auc_lgbm:.4f}")

Train AUROC: 0.9524
Validation AUROC: 0.9356


Отримана модель LightGBM показала хороший результат: AUROC на тренувальній вибірці становить 0.9524, а на валідаційній — 0.9356. Невелика різниця між цими значеннями (0.0168) свідчить про відсутність значного перенавчання моделі.

При порівнянні з базовою моделлю XGBoost, яка мала Validation AUROC = 0.9356, LightGBM показав такий самий результат. Отже, в цьому експерименті LightGBM не дозволив покращити якість класифікації, але продемонстрував таку ж високу якість на валідаційній вибірці.

5. Використовуючи бібліотеку `Hyperopt` і приклад пошуку гіперпараметрів для `LightGBM` з лекції знайдіть оптимальні значення гіперпараметрів `LightGBM` для нашої задачі. Задайте свою сітку гіперпараметрів виходячи з тих параметрів, які ви б хотіли перебрати. Поставте кількість раундів в підборі гіперпараметрів рівну **10**.

  **Увага!** Для того, аби скористатись hyperopt, нам треба задати функцію `objective`. І тут ми також ставимо loss - негативне значення AUROC, як і при пошуці гіперпараметрів для XGBoost. До речі, можна спробувати написати код так, аби в objective передавати лише модель і не писати схожий код двічі :)

  Після успішного завершення пошуку оптимальних гіперпараметрів
    - виведіть найкращі значення гіперпараметрів
    - створіть в окремій зміній `final_lgb_clf` модель `LightGBM` з найкращими гіперпараметрами
    - навчіть модель `final_lgb_clf`
    - оцініть якість моделі `final_lgb_clf` на тренувальній і валідаційній вибірках з допомогою AUROC.
    - зробіть висновок про якість моделі. Чи стала вона краще порівняно з попереднім пунктом (4) цього завдання?

In [25]:
space_lgb = {
    "n_estimators": hp.quniform("n_estimators", 100, 500, 50),
    "learning_rate": hp.uniform("learning_rate", 0.01, 0.2),
    "num_leaves": hp.quniform("num_leaves", 10, 50, 5),
    "max_depth": hp.quniform("max_depth", 3, 10, 1),
    "min_child_samples": hp.quniform("min_child_samples", 10, 50, 5),
    "subsample": hp.uniform("subsample", 0.6, 1.0),
    "colsample_bytree": hp.uniform("colsample_bytree", 0.6, 1.0),
}

In [26]:
def objective_lgb(params):
    params["n_estimators"] = int(params["n_estimators"])
    params["num_leaves"] = int(params["num_leaves"])
    params["max_depth"] = int(params["max_depth"])
    params["min_child_samples"] = int(params["min_child_samples"])

    model = LGBMClassifier(
        **params,
        random_state=42,
        verbosity=-1
    )

    model.fit(
        train_inputs,
        train_targets,
        categorical_feature=cat_feature_indexes
    )

    val_probs = model.predict_proba(val_inputs)[:, 1]

    val_auc = roc_auc_score(
        val_targets,
        val_probs
    )

    return {
        "loss": -val_auc,
        "status": STATUS_OK
    }

In [27]:
trials_lgb = Trials()

best_lgb = fmin(
    fn=objective_lgb,
    space=space_lgb,
    algo=tpe.suggest,
    max_evals=10,
    trials=trials_lgb,
    rstate=np.random.default_rng(42)
)

print("Best hyperparameters:")
print(best_lgb)

100%|██████████| 10/10 [00:06<00:00,  1.49trial/s, best loss: -0.9327498456684272]
Best hyperparameters:
{'colsample_bytree': np.float64(0.9539527896501707), 'learning_rate': np.float64(0.025006823168791832), 'max_depth': np.float64(7.0), 'min_child_samples': np.float64(10.0), 'n_estimators': np.float64(450.0), 'num_leaves': np.float64(35.0), 'subsample': np.float64(0.7201685419162034)}


In [28]:
best_lgb["n_estimators"] = int(best_lgb["n_estimators"])
best_lgb["num_leaves"] = int(best_lgb["num_leaves"])
best_lgb["max_depth"] = int(best_lgb["max_depth"])
best_lgb["min_child_samples"] = int(best_lgb["min_child_samples"])

final_lgb_clf = LGBMClassifier(
    **best_lgb,
    random_state=42,
    verbosity=-1
)

In [29]:
final_lgb_clf.fit(
    train_inputs,
    train_targets,
    categorical_feature=cat_feature_indexes
)

LGBMClassifier(colsample_bytree=np.float64(0.9539527896501707),
               learning_rate=np.float64(0.025006823168791832), max_depth=7,
               min_child_samples=10, n_estimators=450, num_leaves=35,
               random_state=42, subsample=np.float64(0.7201685419162034),
               verbosity=-1)

In [30]:
train_probs_final_lgb = final_lgb_clf.predict_proba(train_inputs)[:, 1]
val_probs_final_lgb = final_lgb_clf.predict_proba(val_inputs)[:, 1]

train_auc_final_lgb = roc_auc_score(
    train_targets,
    train_probs_final_lgb
)

val_auc_final_lgb = roc_auc_score(
    val_targets,
    val_probs_final_lgb
)

print(f"Train AUROC: {train_auc_final_lgb:.4f}")
print(f"Validation AUROC: {val_auc_final_lgb:.4f}")

Train AUROC: 0.9808
Validation AUROC: 0.9327


Після оптимізації гіперпараметрів LightGBM за допомогою Hyperopt Train AUROC становить 0.9808, а Validation AUROC — 0.9327. Порівняно з базовою моделлю LightGBM, Validation AUROC зменшився з 0.9356 до 0.9327.

Отже, оптимізація за допомогою Hyperopt у цьому експерименті не покращила якість моделі на валідаційній вибірці. При цьому Train AUROC значно зріс, а різниця між Train та Validation AUROC стала більшою. Це свідчить про збільшення перенавчання (high variance). Тому базова модель LightGBM у цьому випадку показала кращу здатність до узагальнення.

In [31]:
final_xgb = XGBClassifier(
    n_estimators=400,
    max_depth=3,
    learning_rate=0.025,
    min_child_weight=3,
    subsample=0.8,
    colsample_bytree=0.8,
    gamma=0.05,
    reg_alpha=0,
    reg_lambda=2.0,
    max_bin=256,
    random_state=42,
    enable_categorical=True,
    tree_method="hist"
)

final_xgb.fit(train_inputs, train_targets)

train_probs = final_xgb.predict_proba(train_inputs)[:, 1]
val_probs = final_xgb.predict_proba(val_inputs)[:, 1]

print(
    f"Train AUROC: {roc_auc_score(train_targets, train_probs):.6f}"
)
print(
    f"Validation AUROC: {roc_auc_score(val_targets, val_probs):.6f}"
)

Train AUROC: 0.943360
Validation AUROC: 0.937485


6. Оберіть модель з експериментів в цьому ДЗ і зробіть новий `submission` на Kaggle та додайте код для цього і скріншот скора на публічному лідерборді.
  
  **Напишіть коментар, чому ви обрали саме цю модель?**

  І я вас вітаю - це останнє завдання з цим набором даних 💪 На цьому етапі корисно проаналізувати, які моделі показали себе найкраще і подумати, чому.

In [33]:
test_df = pd.read_csv("test.csv")

test_inputs = test_df[input_cols].copy()

for col in categorical_cols:
    test_inputs[col] = test_inputs[col].astype("category")

full_train_inputs = raw_df[input_cols].copy()

for col in categorical_cols:
    full_train_inputs[col] = full_train_inputs[col].astype("category")

final_xgb.fit(full_train_inputs, raw_df["Exited"])

test_predictions = final_xgb.predict_proba(test_inputs)[:, 1]

submission = pd.DataFrame({
    "id": test_df["id"],
    "Exited": test_predictions
})

submission.to_csv("submission.csv", index=False)

Для нового submission я обрала модель XGBoostClassifier з оптимальними гіперпараметрами, отриманими в результаті експериментів. Саме ця модель показала найкращий результат на валідаційній вибірці — AUROC близько 0.9375. Вона перевершила базові моделі Decision Tree та LightGBM і показала кращу якість на валідаційних даних. Крім того, різниця між Train AUROC та Validation AUROC не є критично великою, тому модель має достатньо хорошу здатність до узагальнення. Для фінального submission я використала цю конфігурацію XGBoost.